# Baselines — epoch 0 and frozen encoder

Two controls requested at Major Updates I. Neither requires retraining anything.

**Epoch 0** — each model evaluated *before* a single weight update, with its freshly
initialised classification head. This is the floor: it shows what the score is when
no learning has happened, so the sweep curves have a starting point.

**Frozen encoder** — the pretrained encoder is locked and used only as a feature
extractor. A logistic regression is trained on the resulting sentence embeddings.
This isolates how much fine-tuning actually contributed, as opposed to what the
pretrained representations already carried.

Together with the TF-IDF baseline and the fine-tuned sweep results, this gives four
points on one scale:

    TF-IDF + logistic regression
    epoch 0 (no training at all)
    frozen encoder + logistic regression
    fully fine-tuned (mean over 3 seeds)

Runtime: roughly 15-25 minutes total on a T4. No training, so this is far cheaper
than the sweep. Results append to `metrics/baseline_results.json` in Drive.

In [ ]:
# ============================================================
# 0. Environment
# ============================================================
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (slower, but this will still finish)")

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
LANGS  = ["tamil", "malayalam"]
MODELS = ["xlm-roberta-base", "google/muril-base-cased"]

MAX_LEN     = 128      # same as the sweep
ENC_BATCH   = 64       # batch size for embedding extraction (inference only)
SEED        = 42
C_GRID      = [0.01, 0.1, 1.0, 10.0]   # logistic-regression strength, selected on DEV

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, gc, time

BASE_DIR     = Path("/content/drive/MyDrive/hope_speech")
DATA_DIR     = BASE_DIR / "processed"
METRICS_OUT  = BASE_DIR / "metrics"
METRICS_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = METRICS_OUT / "baseline_results.json"

def load_results():
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text())
    return {"runs": []}

def save_results(res):
    RESULTS_PATH.write_text(json.dumps(res, indent=2))

def already_done(res, kind, model, lang):
    return any(r["kind"] == kind and r["model"] == model and r["lang"] == lang
               for r in res["runs"])

results = load_results()
print(f"Config OK - {len(results['runs'])} baseline runs already recorded")
print(f"Planned: {len(MODELS) * len(LANGS)} epoch-0 + {len(MODELS) * len(LANGS)} frozen-encoder = {2 * len(MODELS) * len(LANGS)} total")

In [ ]:
# ============================================================
# 2. Data + metrics  (identical to the sweep, so numbers are comparable)
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def load_split(lang, split):
    df = pd.read_csv(DATA_DIR / f"{lang}_{split}_processed.csv")
    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.strip() != ""]
    assert set(df["label"].unique()) <= {0, 1}
    return df[["text", "label"]].reset_index(drop=True)

def scores(y_true, y_pred):
    return {"accuracy":       round(accuracy_score(y_true, y_pred), 4),
            "macro_f1":       round(f1_score(y_true, y_pred, average="macro"), 4),
            "hope_f1":        round(f1_score(y_true, y_pred, pos_label=1), 4),
            "hope_precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "hope_recall":    round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)}

_data = {}
def get_data(lang):
    if lang not in _data:
        _data[lang] = {s: load_split(lang, s) for s in ("train", "dev", "test")}
        n = {s: len(d) for s, d in _data[lang].items()}
        print(f"  {lang}: train {n['train']} | dev {n['dev']} | test {n['test']}")
    return _data[lang]

for lang in LANGS:
    get_data(lang)

In [ ]:
# ============================================================
# 3. Epoch 0  -  the untrained floor
# ============================================================
# A pretrained encoder with a randomly initialised classification head, evaluated
# before any weight update. Seed 42, matching the sweep's grid seed, so this is
# literally the state each sweep run started from.

from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed

@torch.no_grad()
def predict_cls(model, tok, texts):
    model.eval()
    out = []
    for i in range(0, len(texts), ENC_BATCH):
        enc = tok(texts[i:i + ENC_BATCH], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(DEVICE)
        out.append(model(**enc).logits.argmax(-1).cpu().numpy())
    return np.concatenate(out)

def epoch_zero(model_name, lang):
    t0 = time.time()
    dfs = get_data(lang)
    set_seed(SEED)
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)

    rec = {"kind": "epoch0", "model": model_name, "lang": lang, "seed": SEED}
    for split in ("dev", "test"):
        df = dfs[split]
        rec[split] = scores(df["label"].values, predict_cls(model, tok, df["text"].tolist()))
    rec["minutes"] = round((time.time() - t0) / 60, 2)

    del model; gc.collect(); torch.cuda.empty_cache()
    return rec

def short(m):
    return "XLM-R" if "xlm" in m else "MuRIL"

for lang in LANGS:
    for m in MODELS:
        if already_done(results, "epoch0", m, lang):
            print(f"SKIP  epoch0  {short(m):6s} {lang}")
            continue
        print(f"RUN   epoch0  {short(m):6s} {lang} ...")
        r = epoch_zero(m, lang)
        results["runs"].append(r); save_results(results)
        print(f"      dev macro F1 {r['dev']['macro_f1']:.4f} | test {r['test']['macro_f1']:.4f} "
              f"| hope F1 {r['test']['hope_f1']:.4f}")

print("\nEpoch-0 evaluation complete.")

In [ ]:
# ============================================================
# 4. Frozen encoder  -  how much did fine-tuning actually add?
# ============================================================
# The encoder's weights are never updated. Each comment is pushed through once and
# mean-pooled over its real tokens to give a sentence vector. A logistic regression
# is then fitted on those vectors.
#
# Selection discipline matches the sweep: the regularisation strength C is chosen on
# DEV, and TEST is scored once with the winner.

from transformers import AutoModel
from sklearn.linear_model import LogisticRegression

@torch.no_grad()
def embed(encoder, tok, texts):
    encoder.eval()
    vecs = []
    for i in range(0, len(texts), ENC_BATCH):
        enc = tok(texts[i:i + ENC_BATCH], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(DEVICE)
        hidden = encoder(**enc).last_hidden_state              # (B, T, H)
        mask   = enc["attention_mask"].unsqueeze(-1).float()   # (B, T, 1)
        pooled = (hidden * mask).sum(1) / mask.sum(1)          # mean over real tokens
        vecs.append(pooled.float().cpu().numpy())
    return np.vstack(vecs)

def frozen_probe(model_name, lang):
    t0 = time.time()
    dfs = get_data(lang)
    tok     = AutoTokenizer.from_pretrained(model_name)
    encoder = AutoModel.from_pretrained(model_name).to(DEVICE)

    X, y = {}, {}
    for split in ("train", "dev", "test"):
        X[split] = embed(encoder, tok, dfs[split]["text"].tolist())
        y[split] = dfs[split]["label"].values
    del encoder; gc.collect(); torch.cuda.empty_cache()

    # select C on dev
    trials = []
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(X["train"], y["train"])
        dev_f1 = f1_score(y["dev"], clf.predict(X["dev"]), average="macro")
        trials.append({"C": C, "dev_macro_f1": round(dev_f1, 4), "clf": clf})
        print(f"        C={C:<6} dev macro F1 {dev_f1:.4f}")

    best = max(trials, key=lambda t: t["dev_macro_f1"])
    rec = {"kind": "frozen", "model": model_name, "lang": lang, "best_C": best["C"],
           "C_grid": [{k: t[k] for k in ("C", "dev_macro_f1")} for t in trials],
           "dev":  scores(y["dev"],  best["clf"].predict(X["dev"])),
           "test": scores(y["test"], best["clf"].predict(X["test"])),
           "minutes": round((time.time() - t0) / 60, 2)}
    return rec

for lang in LANGS:
    for m in MODELS:
        if already_done(results, "frozen", m, lang):
            print(f"SKIP  frozen  {short(m):6s} {lang}")
            continue
        print(f"RUN   frozen  {short(m):6s} {lang} ...")
        r = frozen_probe(m, lang)
        results["runs"].append(r); save_results(results)
        print(f"      best C={r['best_C']} | dev {r['dev']['macro_f1']:.4f} "
              f"| test {r['test']['macro_f1']:.4f} | {r['minutes']:.1f} min")

print("\nFrozen-encoder baselines complete.")

In [ ]:
# ============================================================
# 5. Summary  -  the four-point comparison for the slide
# ============================================================
# Pulls in the sweep results so epoch 0, frozen and fine-tuned sit on one scale.

sweep_path = METRICS_OUT / "sweep_results.json"
sweep = json.loads(sweep_path.read_text()) if sweep_path.exists() else {"runs": []}

def finetuned_mean(model, lang):
    """Mean test macro F1 over the stage-2 seeds at the dev-selected best config."""
    grid = [r for r in sweep["runs"] if r.get("stage") == "grid" and r["model"] == model
            and r["lang"] == lang and not r.get("degenerate")]
    if not grid:
        return None, None
    best_lr = max(grid, key=lambda r: r["dev_macro_f1"])["lr"]
    runs = [r["test_macro_f1"] for r in sweep["runs"]
            if r["model"] == model and r["lang"] == lang and abs(r["lr"] - best_lr) < 1e-12]
    if not runs:
        return None, None
    return float(np.mean(runs)), float(np.std(runs, ddof=1)) if len(runs) > 1 else 0.0

def get(kind, model, lang):
    for r in results["runs"]:
        if r["kind"] == kind and r["model"] == model and r["lang"] == lang:
            return r
    return None

print("=" * 78)
print("TEST MACRO F1  -  how much does each stage of model capacity buy?")
print("=" * 78)
for lang in LANGS:
    print(f"\n{lang.upper()}")
    print(f"  {'approach':<40} {'test macro F1':>14}")
    print("  " + "-" * 56)
    for m in MODELS:
        e0 = get("epoch0", m, lang)
        fr = get("frozen", m, lang)
        ft, sd = finetuned_mean(m, lang)
        if e0: print(f"  {short(m) + ' - epoch 0 (no training)':<40} {e0['test']['macro_f1']:>14.4f}")
        if fr: print(f"  {short(m) + ' - frozen encoder + LR':<40} {fr['test']['macro_f1']:>14.4f}")
        if ft: print(f"  {short(m) + ' - fine-tuned, mean of 3 seeds':<40} "
                     f"{ft:>9.4f} +/- {sd:.4f}")
        if fr and ft:
            gain = ft - fr["test"]["macro_f1"]
            print(f"  {'':40} {'gain from fine-tuning ' + format(gain, '+.4f'):>14}")
        print()

print("\nReminder for interpretation: on Tamil the seed spread was 0.0342 and on")
print("Malayalam 0.0213. A fine-tuning gain smaller than that is inside the noise.")
print(f"\nSaved -> metrics/baseline_results.json  ({len(results['runs'])} runs)")